# Market Risk Economic Capital Demo

**Economic Capital Simulator** – Market Risk Module  
Ajayvir Khara | Passed FRM Part I and Part II | January 2026

This notebook demonstrates the full Market Risk simulation pipeline:
- Real-world risk factor data download via yfinance
- Covariance calibration (EWMA default, with GARCH comparison)
- 500,000-path Monte Carlo with multivariate Student-t shocks
- 10-day and 1-year VaR/ES at 99.9% confidence
- Euler allocation of Expected Shortfall to individual positions
- Visual breakdown of top contributors

**Expected runtime**: ~60–90 seconds on a standard laptop

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
import os

# Add the project root (parent of notebooks/) to Python path
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

print("Project root added to sys.path:", project_root)
print("Current working directory:", os.getcwd())


# Styling
plt.style.use("seaborn-v0_8")
sns.set_palette("husl")
plt.rcParams["figure.figsize"] = (12, 6)

In [ ]:
# Project imports
from econ_capital.market_risk.data_loaders import (
    load_real_risk_factors,
    load_dummy_positions,
)
from econ_capital.market_risk.engine import MarketRiskEconomicCapital
from econ_capital.market_risk.config import load_market_yaml

## 1. Load Real Historical Risk Factor Returns

In [ ]:
%%time

print("Downloading real risk factor returns (2020–2025)...")
risk_factors = load_real_risk_factors(start="2020-01-01", end="2025-01-01")

print(
    f"Loaded {risk_factors.shape[1]} factors over {risk_factors.shape[0]} trading days"
)
risk_factors.tail()

## 2. Load Sample Portfolio Positions

A stylised £500M+ portfolio across equities, rates, credit, commodities, and FX.

In [ ]:
%%time

positions = load_dummy_positions()
print(f"Portfolio has {positions.shape[0]} positions")
positions.head(10)

## 3. Run Full Monte Carlo Simulation (500,000 paths)

In [ ]:
%%time

engine = MarketRiskEconomicCapital(
    risk_factors=risk_factors,
    positions=positions,
    config=load_market_yaml(),  # Uses market_config.yaml defaults (EWMA, Student-t df=7, etc.)
)

results = engine.run()
print("Market Risk simulation complete!")

## 4. Key Risk Metrics (99.9% Confidence)

In [ ]:
%%time

metrics = {
    "10-day VaR": results["var_10d_999"],
    "10-day ES": results["es_10d_999"],
    "1-year VaR": results["var_1y_999"],
    "1-year ES": results["es_1y_999"],
}

pd.Series(metrics).apply(lambda x: f"£{x:,.0f}")

## 5. Top Position Contributions to 1-Year Expected Shortfall

In [ ]:
%%time

import matplotlib.ticker as mticker

top_10 = results["capital_breakdown"].head(10)

# Convert EC to millions for cleaner display
top_10_millions = top_10 / 1_000_000

plt.figure(figsize=(13, 7))

top_10_millions.plot(kind="barh", color=sns.color_palette("husl", 10))
plt.title(
    "Top 10 Position Contributions to 1-Year Expected Shortfall (99.9%)",
    fontsize=16,
    pad=20,
)
plt.xlabel("Economic Capital (£ Millions)", fontsize=12)
plt.ylabel("")  # Removes default y-label for cleaner look

# Invert so largest contributor is on top
plt.gca().invert_yaxis()

# Grid styling
plt.grid(axis="x", alpha=0.3, linestyle="--")

# Format x-axis: clean £XM labels
plt.gca().xaxis.set_major_formatter(mticker.StrMethodFormatter("{x:,.0f}"))

# Add value labels on the bars
for i, (idx, value) in enumerate(top_10_millions.items()):
    plt.text(
        value + max(top_10_millions) * 0.01,
        i,
        f"£{value:,.1f}M",
        va="center",
        fontsize=10,
        fontweight="bold",
    )

plt.tight_layout()
plt.show()

## 6. Compare Covariance Methods: EWMA vs GARCH

In [ ]:
%%time

from econ_capital.market_risk.covariance import ewma_cov, garch_cov

ewma_vol = np.sqrt(252 * np.diag(ewma_cov(risk_factors, lamb=0.97)))
garch_vol = np.sqrt(252 * np.diag(garch_cov(risk_factors)))

vol_df = pd.DataFrame(
    {"EWMA": ewma_vol, "GARCH": garch_vol}, index=risk_factors.columns
)

vol_df.plot(kind="bar", figsize=(12, 6))
plt.title("Annualized Factor Volatility: EWMA vs GARCH")
plt.ylabel("Volatility")
plt.xticks(rotation=45)
plt.grid(axis="y", alpha=0.3)
plt.show()

## 7. Open the Generated Regulatory-Grade Excel Report

In [ ]:
%%time

import os
import glob

report_pattern = os.path.join(
    project_root,
    "econ_capital",
    "market_risk",
    "reports",
    "MarketRisk_EC_Report_*.xlsx",
)
reports = glob.glob(report_pattern)

if reports:
    latest_report = max(reports, key=os.path.getctime)
    print(f"Opening latest report: {os.path.basename(latest_report)}")
    os.startfile(latest_report)  # Windows
else:
    print("No Market Risk report found.")
    reports_dir = os.path.dirname(report_pattern)
    print(f"Checked directory: {reports_dir}")
    if os.path.exists(reports_dir):
        print("Files in directory:")
        print(os.listdir(reports_dir))
    else:
        print("Reports directory does not exist!")

## Next Steps

- Edit `market_config.yaml` to try `cov_method: "GARCH"` or change `df_t`
- Modify positions in `load_dummy_positions()`
- Run the full firm-wide simulation:  
  ```bash
  python econ_capital/run_full_ec.py
  ```

See the other notebooks:
- `demo_credit.ipynb`
- `demo_oprisk.ipynb`